# Phase I Apple App Store Ingestion Validation
## Tooling Incompatibility & Dependency Friction Report

### Objective
This notebook documents the empirical validation of the `app-store-scraper` library within the standard Sciencia AI development environment. 
The objective was to verify if the Apple App Store toolchain meets the criteria for rapid deployment in Phase I infrastructure, specifically focusing on environment stability and dependency management overhead.

In [1]:
!pip install app-store-scraper

  Attempting uninstall: chardet
    Found existing installation: chardet 4.0.0
    Uninstalling chardet-4.0.0:
      Successfully uninstalled chardet-4.0.0
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.3.0
    Uninstalling urllib3-2.3.0:
      Successfully uninstalled urllib3-2.3.0
  Attempting uninstall: idna━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
    Found existing installation: idna 3.7━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
    Uninstalling idna-3.7:╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
      Successfully uninstalled idna-3.7m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
  Attempting uninstall: requests8;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
    Found existing installation: requests 2.32.38;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
    Uninstalling requests-2.32.3:;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib3]
      Successfully uninstalled requests-2.32.3━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [urllib

In [2]:
from app_store_scraper import AppStore

ModuleNotFoundError: No module named 'urllib3.packages.six.moves'

Apple library ecosystem has dependency stability risks in the current environment.

In [1]:
import requests
print(requests.__version__)

ModuleNotFoundError: No module named 'urllib3.packages.six.moves'

### 🚨 Critical Failure Analysis & Engineering Implication

* **Dependency Conflict Triggered:** Installing `app-store-scraper==0.3.5` automatically forced a massive downgrade of core networking libraries (`requests` downgraded to `2.23.0` and `urllib3` to `1.25.11`).
* **Environment Breaking:** As shown in the tracebacks above, this downgrade immediately broke `urllib3` inside Python 3.13 (`ModuleNotFoundError: No module named 'urllib3.packages.six.moves'`), rendering the HTTP library completely non-functional.
* **System-Wide Blast Radius:** The standard Anaconda package resolver instantly flagged critical compatibility conflicts with enterprise-essential tools currently running in our stack, including `conda`, `streamlit`, `sphinx`, and `jupyterlab-server`.

> **Engineering Implication:** > The current mainstream open-source tooling for Apple App Store scraping is fundamentally **antiquated and incompatible** with modern Python 3.13+ environments. Introducing this package directly into a shared repository or a unified pipeline will destabilize existing data systems and introduce severe dependency friction, making it a critical risk for Phase I.

## 2. Environment Disaster Recovery & Rollback Execution
Objective: Execute a complete purge of the conflicting legacy toolchain and force-restore the modern production-ready network stack to return the Anaconda environment to a baseline stable state.

In [2]:
!pip uninstall -y requests urllib3 chardet idna app-store-scraper

Found existing installation: requests 2.23.0
Uninstalling requests-2.23.0:
  Successfully uninstalled requests-2.23.0
Found existing installation: urllib3 1.25.11
Uninstalling urllib3-1.25.11:
  Successfully uninstalled urllib3-1.25.11
Found existing installation: chardet 3.0.4
Uninstalling chardet-3.0.4:
  Successfully uninstalled chardet-3.0.4
Found existing installation: idna 2.10
Uninstalling idna-2.10:
  Successfully uninstalled idna-2.10
Found existing installation: app-store-scraper 0.3.5
Uninstalling app-store-scraper-0.3.5:
  Successfully uninstalled app-store-scraper-0.3.5


In [3]:
!pip install "requests>=2.31.0" "urllib3>=2.0" "chardet>=5.0" "idna>=3.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 854.0/854.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [requests]


In [4]:
import requests
import urllib3

print("requests:", requests.__version__)
print("urllib3:", urllib3.__version__)

requests: 2.34.2
urllib3: 2.7.0


### Environment Mitigation Findings & Engineering Implication
* **Recovery Status:** Successfully restored the environment to `requests 2.34.2` and `urllib3 2.7.0`, removing the runtime crashes.

> **Engineering Implication (Deployment Guardrails):**
> If Apple App Store integration becomes an absolute business requirement for Phase II, it **cannot** be co-located within our main service architecture. It must be strictly isolated inside a dedicated **Docker Container** running an older Python base image (e.g., Python 3.9/3.10) with locked legacy dependencies, completely decoupled from our primary sentiment analytics infrastructure via an asynchronous message broker (e.g., RabbitMQ or AWS SQS).

## 3. Phase I Architectural Decision (ADR)

* **Status:** **REJECTED for Phase I.**
* **Technical Justification:** Google Play tooling (`google-play-scraper`) runs natively under Python 3.13 with zero dependency conflict, supporting cursor-based pagination out-of-the-box. Conversely, Apple App Store tooling introduces environment corruption risks, high maintenance overhead, and unvalidated pagination states.
* **Next Steps:** Lock the current clean state of this notebook as empirical proof for the architecture review session with John. Focus 100% of Phase I engineering bandwidth on scaling the Google Play Store data ingestion pipeline.